In [ ]:
import shap
import lime
import lime.lime_tabular
import dice_ml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def analizar_prediccion_completo(modelo, X_train_scaled, X_test_scaled, features_names, y_train, idx_muestra=0, class_names=['Fake', 'Real']):

    # 1. Preparación de DataFrames 
    X_train_df = pd.DataFrame(X_train_scaled, columns=features_names)
    X_test_df = pd.DataFrame(X_test_scaled, columns=features_names)
    instance = X_test_df.iloc[[idx_muestra]]
    
    print(f"--- ANALIZANDO MUESTRA ÍNDICE: {idx_muestra} ---")
    pred = modelo.predict(instance)[0]
    prob = modelo.predict_proba(instance)[0]
    print(f"Predicción del modelo: {class_names[int(pred)]} (Confianza: {prob[int(pred)]:.2f})")
    print("-" * 50)

    # 2. SHAP 
    print("\n[1/3] Generando SHAP Local (KernelExplainer)...")
    # Usamos KernelExplainer con un resumen del entrenamiento para mayor velocidad
    explainer_shap = shap.KernelExplainer(modelo.predict_proba, shap.sample(X_train_df, 50))
    shap_values = explainer_shap.shap_values(instance)
    
    shap.initjs()
    # Visualizamos la fuerza de contribución para la clase predicha
    display(shap.force_plot(
        explainer_shap.expected_value[int(pred)], 
        shap_values[int(pred)], 
        instance
    ))

    # 3. LIME
    print("\n[2/3] Generando LIME...")
    explainer_lime = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_df.values,
        feature_names=features_names,
        class_names=class_names,
        mode='classification'
    )
    exp = explainer_lime.explain_instance(
        X_test_df.iloc[idx_muestra].values, 
        modelo.predict_proba,
        num_features=10
    )
    exp.show_in_notebook(show_table=True)

    # 4. DiCE 
    print("\n[3/3] Generando ejemplos contrafácticos con DiCE...")
    df_dice = X_train_df.copy()
    df_dice['label'] = y_train.values
    
    d = dice_ml.Data(
        dataframe=df_dice, 
        continuous_features=features_names, 
        outcome_name='label'
    )
    m = dice_ml.Model(model=modelo, backend="sklearn")
    exp_dice = dice_ml.Dice(d, m, method="random")
    
    dice_exp = exp_dice.generate_counterfactuals(
        instance, 
        total_CFs=2, 
        desired_class="opposite"
    )
    dice_exp.visualize_as_dataframe(show_only_changes=True)


nombres_caracteristicas = X.columns.tolist()

analizar_prediccion_completo(
    modelo=model, 
    X_train_scaled=X_train_scaled, 
    X_test_scaled=X_test_scaled, 
    features_names=nombres_caracteristicas, 
    y_train=y_train,
    idx_muestra=0
)

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
